# Skyntri — Skin / Non-skin Gate (EfficientNet, binary)

This small model decides whether a photo is **skin at all**, before any skin analysis runs. It replaces the built-in rule-based check, which can make mistakes in unusual lighting or with skin-coloured objects. **It has not been run.** Run it on Colab **after** notebooks 01 and 02, because it reuses their cleaned skin images.

| | Data | Link |
|---|---|---|
| **Skin (positive)** | cleaned images from notebooks 01 (faces) and 02 (face and body skin) | your Drive `work/` folders |
| **Not skin (negative)** | **COCO 2017** photos **without people**: phones, laptops, keyboards, TVs, food, animals, rooms, vehicles… | https://cocodataset.org/#download (images: http://images.cocodataset.org/zips/val2017.zip, labels: http://images.cocodataset.org/annotations/annotations_trainval2017.zip) |
| Not skin (textures) | **DTD**, Describable Textures Dataset | https://www.robots.ox.ac.uk/~vgg/data/dtd/ (https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz) |
| Not skin (your own) | optional folder with screenshots, documents, or anything users uploaded by mistake | `DATA_ROOT/extra_negatives/` |

COCO photos that contain a **person** are removed, because people show skin. The thresholds are chosen so that, on validation data, **at most 1% of non-skin photos are accepted** and **at most 1% of skin photos are rejected**. Photos in between get "please upload another photo".

The export is `skin_gate.keras` and `skin_gate_metadata.json`. Copy both to **`backend/models/`**; the backend (`backend/app/services/image_gate.py`) uses them automatically.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip -q install "tensorflow==2.20.0" scikit-learn pandas matplotlib tqdm

In [ ]:
import hashlib
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

SEED = 42
DRIVE = Path("/content/drive/MyDrive/skyntri")
DATA_ROOT = DRIVE / "datasets" / "gate"
WORK_DIR = DRIVE / "work" / "gate"
EXPORT_DIR = DRIVE / "export"
SKIN_SOURCES = {  # cleaned caches written by notebooks 01 and 02
    "facial": DRIVE / "work" / "facial",
    "disease": DRIVE / "work" / "disease",
}
COCO_ROOT = DATA_ROOT / "coco"        # val2017/ + annotations/instances_val2017.json
DTD_ROOT = DATA_ROOT / "dtd"          # dtd/images/<texture>/*.jpg
EXTRA_NEG = DATA_ROOT / "extra_negatives"

IMG_SIZE = 224                        # EfficientNetB0
BATCH_SIZE = 64
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 8
PRETRAINED_WEIGHTS = "imagenet"
NEG_PER_POS = 1.5                     # class balance
TARGET_FALSE_ACCEPT = 0.01            # <= 1% of non-skin accepted (validation)
TARGET_SKIN_REJECT = 0.01             # <= 1% of skin rejected (validation)
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
for p in (DATA_ROOT, WORK_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

## 1. Download negatives (COCO val2017 is about 1 GB, DTD about 600 MB)

In [ ]:
if IN_COLAB and not (COCO_ROOT / "annotations" / "instances_val2017.json").exists():
    COCO_ROOT.mkdir(parents=True, exist_ok=True)
    !wget -q -O /content/val2017.zip http://images.cocodataset.org/zips/val2017.zip
    !wget -q -O /content/ann2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q /content/val2017.zip -d "{COCO_ROOT}"
    !unzip -q -o /content/ann2017.zip "annotations/instances_val2017.json" -d "{COCO_ROOT}"

if IN_COLAB and not DTD_ROOT.exists():
    DTD_ROOT.mkdir(parents=True, exist_ok=True)
    !wget -q -O /content/dtd.tar.gz https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz
    !tar -xzf /content/dtd.tar.gz -C "{DTD_ROOT}"

## 2. Build the manifest

In [ ]:
rows = []

# Positives: cleaned skin images from notebooks 01/02 (already de-duplicated)
for name, work in SKIN_SOURCES.items():
    manifest_file = work / "manifest_clean.pkl"
    if manifest_file.exists():
        m = pd.read_pickle(manifest_file)
        for _, r in m.iterrows():
            rows.append({"path": r["clean_path"], "label": 1, "category": f"skin:{name}", "group": f"{name}:{r['group']}"})
    else:
        print("Missing (run notebook first):", manifest_file)

# Negatives: COCO images with NO person, category = COCO supercategory of the largest object
ann_file = COCO_ROOT / "annotations" / "instances_val2017.json"
if ann_file.exists():
    coco = json.loads(ann_file.read_text())
    cats = {c["id"]: c for c in coco["categories"]}
    person_imgs, biggest = set(), {}
    for a in coco["annotations"]:
        if cats[a["category_id"]]["name"] == "person":
            person_imgs.add(a["image_id"])
        if a["area"] > biggest.get(a["image_id"], (0, ""))[0]:
            biggest[a["image_id"]] = (a["area"], cats[a["category_id"]]["supercategory"])
    for im in coco["images"]:
        if im["id"] in person_imgs:
            continue
        p = COCO_ROOT / "val2017" / im["file_name"]
        if p.exists():
            rows.append({"path": str(p), "label": 0, "category": f"coco:{biggest.get(im['id'], (0, 'scene'))[1]}",
                         "group": f"coco:{im['id']}"})

for p in DTD_ROOT.rglob("*.jpg"):
    rows.append({"path": str(p), "label": 0, "category": "dtd:texture", "group": f"dtd:{p.stem}"})
for p in EXTRA_NEG.rglob("*") if EXTRA_NEG.exists() else []:
    if p.suffix.lower() in IMG_EXT:
        rows.append({"path": str(p), "label": 0, "category": "extra", "group": f"extra:{p.stem}"})

manifest = pd.DataFrame(rows)
pos, neg = manifest[manifest.label == 1], manifest[manifest.label == 0]
neg = neg.sample(min(len(neg), int(NEG_PER_POS * len(pos))), random_state=SEED)
manifest = pd.concat([pos, neg]).reset_index(drop=True)
print(manifest["label"].value_counts().rename({1: "skin", 0: "not skin"}))
display(manifest["category"].value_counts())

In [ ]:
def readable(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False


manifest = manifest[[readable(p) for p in tqdm(manifest["path"], desc="checking")]].reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = list(sgkf.split(manifest, manifest["label"], manifest["group"]))
manifest["split"] = "train"
manifest.loc[folds[0][1], "split"] = "test"
manifest.loc[folds[1][1], "split"] = "val"
display(manifest.groupby(["split", "label"]).size().unstack())
train_df, val_df, test_df = (manifest[manifest.split == s].reset_index(drop=True) for s in ("train", "val", "test"))

## 3. Pipeline, model and training

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15, fill_mode="reflect"),
    tf.keras.layers.RandomZoom((-0.25, 0.25), fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomBrightness(0.25, value_range=(0, 255)),   # lighting robustness is the goal here
])


def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear"), tf.float32)


def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values.astype("float32")))
    if training:
        ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)


def build_model(weights=PRETRAINED_WEIGHTS):
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.GlobalAveragePooling2D()(base(inputs, training=False))
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="p_skin")(x)  # single output = P(skin)
    return tf.keras.Model(inputs, outputs, name="skyntri_skin_gate"), base


model, base = build_model()
LOSS = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05)
METRICS = [tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.BinaryAccuracy(name="acc")]
CKPT = WORK_DIR / "best_gate.keras"
cbs = lambda patience: [  # noqa: E731
    tf.keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_auc", mode="max", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=patience, restore_best_weights=True),
]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=cbs(2))
base.trainable = True
for layer in base.layers[: int(len(base.layers) * 0.5)]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(3e-5), loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=cbs(3))

## 4. Thresholds (validation) and test evaluation

In [ ]:
best = tf.keras.models.load_model(CKPT, compile=False)


def probs(ds):
    return np.concatenate([best.predict_on_batch(x).reshape(-1) for x, _ in ds])


p_val, y_val = probs(val_ds), val_df["label"].values
neg_val, pos_val = np.sort(p_val[y_val == 0]), np.sort(p_val[y_val == 1])
# accept: at most TARGET_FALSE_ACCEPT of non-skin reach it
accept_threshold = float(np.quantile(neg_val, 1 - TARGET_FALSE_ACCEPT)) if len(neg_val) else 0.8
# reject: at most TARGET_SKIN_REJECT of skin fall below it
reject_threshold = float(np.quantile(pos_val, TARGET_SKIN_REJECT)) if len(pos_val) else 0.4
accept_threshold = float(np.clip(accept_threshold, 0.05, 0.99))
reject_threshold = float(min(np.clip(reject_threshold, 0.01, 0.95), accept_threshold))
print("accept >=", round(accept_threshold, 4), "| reject <", round(reject_threshold, 4))

p_test, y_test = probs(test_ds), test_df["label"].values
accepted, rejected = p_test >= accept_threshold, p_test < reject_threshold
uncertain = ~accepted & ~rejected


def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(max(0, c - h), 4), round(min(1, c + h), 4))


neg, pos = y_test == 0, y_test == 1
summary = {
    "roc_auc": roc_auc_score(y_test, p_test),
    "non_skin_false_accept_rate": float(accepted[neg].mean()),
    "non_skin_false_accept_ci95": wilson(int(accepted[neg].sum()), int(neg.sum())),
    "skin_rejection_rate": float(rejected[pos].mean()),
    "skin_rejection_ci95": wilson(int(rejected[pos].sum()), int(pos.sum())),
    "skin_accepted_rate": float(accepted[pos].mean()),
    "uncertain_rate": float(uncertain.mean()),
}
display(pd.Series(summary))

per_cat = pd.DataFrame([
    {"category": c, "n": len(i), "accepted_as_skin": float(accepted[i].mean()), "uncertain": float(uncertain[i].mean())}
    for c, i in test_df.groupby("category").indices.items()]).sort_values("accepted_as_skin", ascending=False)
display(per_cat)

fpr, tpr, _ = roc_curve(y_test, p_test)
plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "k--"); plt.title(f"Skin gate ROC (AUC {summary['roc_auc']:.3f})")
plt.xlabel("false accept rate"); plt.ylabel("skin acceptance"); plt.show()

worst = test_df[neg].assign(p=p_test[neg]).sort_values("p", ascending=False).head(8)
fig, axes = plt.subplots(1, len(worst), figsize=(3 * max(1, len(worst)), 3))
for ax, (_, r) in zip(np.atleast_1d(axes), worst.iterrows()):
    ax.imshow(Image.open(r["path"]).convert("RGB")); ax.axis("off"); ax.set_title(f"{r['category']}\nP={r['p']:.2f}", fontsize=8)
plt.suptitle("Non-skin photos the gate likes most (check these)"); plt.show()

## 5. Export and prediction (same as backend `image_gate.py`)

In [ ]:
MODEL_FILE = EXPORT_DIR / "skin_gate.keras"
META_FILE = EXPORT_DIR / "skin_gate_metadata.json"
export_model, _ = build_model()  # ImageNet build = same layers as in training (weights=None drops a Rescaling layer)
export_model.set_weights(best.get_weights())
export_model.save(MODEL_FILE)
META_FILE.write_text(json.dumps({
    "task": "skin_gate",
    "image_size": IMG_SIZE,
    "output": "sigmoid P(skin)",
    "accept_threshold": accept_threshold,
    "reject_threshold": reject_threshold,
    "sha256": hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest(),
    "test_metrics": summary,
    "per_category": per_cat.round(4).to_dict(orient="records"),
}, indent=2, default=float))
print("Saved", MODEL_FILE, META_FILE)

loaded = tf.keras.models.load_model(MODEL_FILE, compile=False)


def is_skin(path):
    with Image.open(path) as im:
        arr = np.asarray(ImageOps.exif_transpose(im).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR), np.float32)
    p = float(loaded.predict(arr[None], verbose=0)[0, 0])
    return {"p_skin": round(p, 4),
            "decision": "accepted" if p >= accept_threshold else "rejected" if p < reject_threshold else "uncertain"}


is_skin(test_df.loc[0, "path"])